In [19]:
import polars as pl
import polars.selectors as cs
import numpy as np
from pathlib import Path
import warnings
import os
import glob
import re
import polars.functions as F
import shutil
import tempfile
import uuid
import io
from concurrent.futures import ThreadPoolExecutor, as_completed

from polars import LazyFrame
from datetime import time, datetime as dt

import pandas as pd

# Suppress unnecessary warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [20]:
# --- Configuration ---
# Define root paths
first_glob = os.path.expanduser("~").replace("\\", "/")
test_path = f"{first_glob}/Concentrix Corporation"
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Not found the path: {test_path}")

# Define folder paths
# *** NOTE: UPDATE "hc_extend" PATH IF NEEDED ***
folder_paths = {
    "rta_output": Path(f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/STORAGE_OUTPUT_RTA'),
    "resources": Path(f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/BI_Task/CODE/Resources'),
    "hc_extend": Path(f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Headcount/HC Extend by Month'),
    "attendance": Path(f'{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Attendance')
}

print("--- FULL FOLDER PATHS LIST ---")
for key, path in folder_paths.items():
    print(f"{key}: {path}")

print("-" * 60)

# --- Constants for Leave Loading ---
LEAVE_REQ_COLS = ["Date","IEX","Leave","Half Shift","Agent Name","Wave","Shift","Final Shift","Reason","Remark","Source_File"]
LEAVE_RENAME_VARIANTS = {
    "Date": ["Leave Date","DATE","Date/Time","Join Date (VNT)","_Date","_Date_Converted"],
    "IEX": ["IEX ID","IEX_Id","IEX Id","IEXID"],
    "Leave": ["Leave Type","LeaveType"],
    "Half Shift": ["Half-Shift","HalfShift"],
    "Agent Name": ["Employee Name","Agent","Name"],
    "Wave": ["Team","Wave Name"],
    "Shift": ["Planned Shift","Schedule"],
    "Final Shift": ["FinalShift","Final_Shift"],
    "Reason": ["Reasons","Reason Code"],
    "Remark": ["Remarks","Comments","Comment"],
}
LEAVE_SHEET_ALIASES = [
    "Leave | Add D\u00e2t","Leave | Add D\u00e1t","Leave|Add D\u00e2t","Leave|Add D\u00e1t",
    "Leave | Add Dat","Leave|Add Dat","Leave | Add Data","Leave|Add Data","Leave"
]

# --- Constants for Roster Loading ---
ROSTER_SHEET="Roster_Rawdata"
ROSTER_ID_COLS=["IEX","Emp ID","Email"]
ROSTER_DATE_RX=re.compile(r"^\s*\d{1,2}/\d{1,2}/\d{2,4}\s*$")

--- FULL FOLDER PATHS LIST ---
rta_output: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\STORAGE_OUTPUT_RTA
resources: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\BI_Task\CODE\Resources
hc_extend: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Headcount\HC Extend by Month
attendance: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Attendance
------------------------------------------------------------


In [21]:
LEAVE_REQ_COLS = ["Date", "IEX", "Leave", "Half Shift", "Agent Name", "Wave", "Shift", "Final Shift", "Reason", "Remark", "Source_File"]
LEAVE_RENAME_VARIANTS = {
    "Date": ["Leave Date", "DATE", "Date/Time", "Join Date (VNT)", "_Date", "_Date_Converted"],
    "IEX": ["IEX ID", "IEX_Id", "IEX Id", "IEXID"],
    "Leave": ["Leave Type", "LeaveType"],
    "Half Shift": ["Half-Shift", "HalfShift"],
    "Agent Name": ["Employee Name", "Agent", "Name"],
    "Wave": ["Team", "Wave Name"],
    "Shift": ["Planned Shift", "Schedule"],
    "Final Shift": ["FinalShift", "Final_Shift"],
    "Reason": ["Reasons", "Reason Code"],
    "Remark": ["Remarks", "Comments", "Comment"],
}
LEAVE_SHEET_ALIASES = ["Leave | Add D\u00e2t", "Leave | Add D\u00e1t", "Leave | Add Dat", "Leave | Add Data", "Leave"]

EMPTY_LEAVE_SCHEMA = {
    "Date": pl.Date, "IEX": pl.Int64, "Leave": pl.String, "Half Shift": pl.String,
    "Agent Name": pl.String, "Wave": pl.String, "Shift": pl.String,
    "Final Shift": pl.String, "Reason": pl.String, "Remark": pl.String,
    "Source_File": pl.String
}

# --- OPTIMIZATION: Precompile header keyword set ---
_HEADER_KEYWORDS = {"DATE", "IEX", "LEAVE", "SHIFT", "AGENT", "NAME", "TIME"}

def _read_excel_with_anti_lock(fp: str) -> pl.DataFrame | None:
    file_name = os.path.basename(fp)
    print(f"  [Anti-Lock] Processing file: {file_name}")

    buffer = None
    temp_path = os.path.join(tempfile.gettempdir(), f"temp_leave_{uuid.uuid4().hex[:8]}.xlsx")

    try:
        with open(fp, "rb") as f:
            buffer = io.BytesIO(f.read())
    except PermissionError:
        try:
            shutil.copyfile(fp, temp_path)
            with open(temp_path, "rb") as f:
                buffer = io.BytesIO(f.read())
        except Exception:
            print(f"  \u274c Strict OS lock detected. Please click outside the active cell or close Excel for this file.")
            return None
    except Exception as e:
        print(f"  \u274c Read error: {e}")
        return None
    finally:
        if os.path.exists(temp_path):
            try:
                os.remove(temp_path)
            except:
                pass

    if not buffer:
        return None

    try:
        try:
            xls = pd.ExcelFile(buffer, engine='calamine')
            available_sheets = xls.sheet_names
            print(f"  -> Sheets found: {available_sheets}")
        except Exception:
            xls = pd.ExcelFile(buffer, engine='openpyxl')
            available_sheets = xls.sheet_names
            print(f"  -> Sheets found (openpyxl): {available_sheets}")

        target_sheet = None
        for sh in LEAVE_SHEET_ALIASES:
            if sh in available_sheets:
                target_sheet = sh
                break

        if not target_sheet:
            for sh in available_sheets:
                if "leave" in sh.lower():
                    target_sheet = sh
                    break

        if not target_sheet:
            print(f"  \u274c No Leave-related sheets found.")
            return None

        print(f"  -> Extracting from: '{target_sheet}'")
        df_pd = pd.read_excel(buffer, sheet_name=target_sheet, engine=xls.engine, header=None, dtype=str)

        # OPTIMIZATION: avoid row iteration — use vectorized string matching on first 20 rows
        best_row_idx = -1
        max_matches = 0
        head20 = df_pd.head(20).fillna("")
        for idx in range(len(head20)):
            row_upper = " ".join(str(x).upper() for x in head20.iloc[idx] if x)
            matches = sum(1 for kw in _HEADER_KEYWORDS if kw in row_upper)
            if matches > max_matches:
                max_matches = matches
                best_row_idx = idx

        if best_row_idx != -1 and max_matches >= 2:
            df_pd.columns = df_pd.iloc[best_row_idx]
            df_pd = df_pd.iloc[best_row_idx + 1:].reset_index(drop=True)
            df_pd = df_pd.loc[:, df_pd.columns.notna()]
            print(f"  -> \u2705 Header found at row {best_row_idx + 1}")
            return pl.from_pandas(df_pd)
        else:
            print(f"  -> \u274c Valid Header not found.")
            return None

    except Exception as e:
        print(f"  \u274c Buffer process error: {e}")
        return None


def standardize_and_keep(df: pl.DataFrame) -> pl.DataFrame:
    df = df.rename({c: c.strip() for c in df.columns})
    for target, alts in LEAVE_RENAME_VARIANTS.items():
        if target not in df.columns:
            for alt in alts:
                if alt in df.columns:
                    df = df.rename({alt: target})
                    break
    for c in LEAVE_REQ_COLS:
        if c not in df.columns:
            df = df.with_columns(pl.lit(None).alias(c))

    # OPTIMIZATION: replace row-by-row map_elements with Polars native chained str.to_date
    if "Date" in df.columns:
        date_col = pl.col("Date").cast(pl.String).str.strip_chars()
        df = df.with_columns(
            pl.coalesce(
                date_col.str.to_date("%Y-%m-%d", strict=False),
                date_col.str.to_date("%d/%m/%Y", strict=False),
                date_col.str.to_date("%m/%d/%Y", strict=False),
                date_col.str.to_date("%d-%m-%Y", strict=False),
                date_col.str.to_datetime(strict=False).dt.date(),
            ).alias("Date")
        )
    return df.select(LEAVE_REQ_COLS)


def load_leave_add_dat(folder: str) -> pl.DataFrame:
    print(f"Loading Leave data from: {folder}")
    patterns = [
        os.path.join(folder, "Expedia - ATD Report*.xlsx"),
        os.path.join(folder, "Expedia - ATD Report*.xlsm"),
        os.path.join(folder, "Expedia - ATD Report*.xls"),
    ]
    files = [f for p in patterns for f in glob.glob(p) if not os.path.basename(f).startswith("~$")]
    if not files:
        return pl.DataFrame(schema=EMPTY_LEAVE_SCHEMA)

    dfs = []
    for f in files:
        df = _read_excel_with_anti_lock(f)
        if df is None:
            continue
        df = df.with_columns(pl.lit(os.path.basename(f)).alias("Source_File"))
        df = standardize_and_keep(df)
        dfs.append(df)

    if not dfs:
        return pl.DataFrame(schema=EMPTY_LEAVE_SCHEMA)
    try:
        out = pl.concat(dfs, how="vertical_relaxed")
    except Exception:
        out = pl.concat(dfs, how="diagonal")

    out = out.with_columns([
        pl.col("IEX").cast(pl.Float64, strict=False).cast(pl.Int64, strict=False).alias("IEX"),
        pl.col("Date").cast(pl.Date, strict=False).alias("Date"),
    ])
    out = out.filter(pl.col("IEX").is_not_null() & pl.col("Date").is_not_null())
    if out.height == 0:
        return pl.DataFrame(schema=EMPTY_LEAVE_SCHEMA)
    out = out.unique(subset=["Date", "IEX"], keep="last", maintain_order=True)
    return out


LEAVE_ADD_DAT = load_leave_add_dat(str(folder_paths["attendance"]))

if len(LEAVE_ADD_DAT) > 0:
    LEAVE_ADD_DAT.write_parquet(folder_paths["resources"] / "leave_data.parquet")
    print("-" * 60)
    print(f"\u2705 SUCCESS: {len(LEAVE_ADD_DAT)} rows loaded.")
else:
    print("-" * 60)
    print("\u26a0\ufe0f NOTICE: No data found. Empty frame created.")

Loading Leave data from: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Attendance
  [Anti-Lock] Processing file: Expedia - ATD Report - 2025.01.xlsx
  -> Sheets found: ['Date', 'Summary', 'Realtime', 'EOD', 'Late Data', 'Sheet5', 'Realtime Data', 'Leave | Add Data', 'Consecutive Absence', 'Roster_Rawdata', 'Activity', 'ATD_Final', 'HC_Extend', 'Sheet2', 'validation', 'Sheet1']
  -> Extracting from: 'Leave | Add Data'
  -> ✅ Header found at row 1
  [Anti-Lock] Processing file: Expedia - ATD Report - 2025.03.xlsx
  -> Sheets found: ['Date', 'Summary', 'EOD', 'Realtime', 'Late Data', 'Sheet5', 'Realtime Data', 'Leave | Add Data', 'Consecutive Absence', 'Roster_Rawdata', 'Activity', 'ATD_Final', 'HC_Extend', 'Sheet2', 'validation', 'Sheet1']
  -> Extracting from: 'Leave | Add Data'
  -> ✅ Header found at row 1
  [Anti-Lock] Processing file: Expedia - ATD Report - 2025.04.xlsx
  -> Sheets found: ['Date', 'Summary', 'EOD', 'Leave | Add Data', 'Roster_Rawdat

In [22]:
master_roster_path = f"{first_glob}/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Schedule/Schedule (Ops version)/2026/Master_Schedule_Merged.xlsx"

print(f"Loading Roster data from: {master_roster_path}")

df = pl.read_excel(master_roster_path)

if "Oracle" in df.columns and "OracleID" not in df.columns:
    df = df.rename({"Oracle": "OracleID"})
if "IEX" in df.columns and "IEX ID" not in df.columns:
    df = df.rename({"IEX": "IEX ID"})

id_cols = ["IEX ID", "OracleID", "Email"]
date_cols = [col for col in df.columns if col not in ["IEX ID", "OracleID", "Email", "Employee Name", "Status", "Termination"]]

combined = df.select(id_cols + date_cols).unpivot(
    index=id_cols,
    on=date_cols,
    variable_name="Scheduled_Date",
    value_name="Shift"
)

combined = combined.with_columns([
    pl.col("Scheduled_Date").cast(pl.String).str.slice(0, 10).str.to_date("%Y-%m-%d", strict=False),
    pl.col("IEX ID").cast(pl.String, strict=False).str.replace_all(r"\D+", "").cast(pl.Int64, strict=False),
    pl.col("OracleID").cast(pl.String, strict=False).str.replace_all(r"\D+", "").cast(pl.Int64, strict=False),
    pl.col("Email").cast(pl.String, strict=False),
    pl.lit(os.path.basename(master_roster_path)).alias("Source_File")
])

trimmed_shift = pl.col("Shift").cast(pl.String, strict=False).str.strip_chars()

combined = combined.filter(
    pl.col("IEX ID").is_not_null() &
    pl.col("Shift").is_not_null() &
    ~trimmed_shift.is_in(["", "-", "0"]) &
    pl.col("Scheduled_Date").is_not_null()
)

# OPTIMIZATION: sort then unique in one lazy pass instead of two separate eager ops
if combined.height > 0:
    combined = (
        combined.lazy()
        .sort(["IEX ID", "Scheduled_Date"])
        .unique(subset=["IEX ID", "Scheduled_Date"], keep="first", maintain_order=True)
        .collect()
    )

combined = combined.select([
    "Scheduled_Date",
    "IEX ID",
    "OracleID",
    "Email",
    "Shift",
    "Source_File"
])

combined.write_parquet(folder_paths["resources"] / "roster_raw.parquet")
print(f"Roster data loaded: {len(combined)} rows")
combined

Loading Roster data from: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Schedule/Schedule (Ops version)/2026/Master_Schedule_Merged.xlsx
Roster data loaded: 13468 rows


Scheduled_Date,IEX ID,OracleID,Email,Shift,Source_File
date,i64,i64,str,str,str
2026-01-05,3000182,102484482,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Master_Schedule_Merged.xlsx"""
2026-01-06,3000182,102484482,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Master_Schedule_Merged.xlsx"""
2026-01-07,3000182,102484482,"""thiphuonghoa.hoang@concentrix.…","""OFF""","""Master_Schedule_Merged.xlsx"""
2026-01-08,3000182,102484482,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Master_Schedule_Merged.xlsx"""
2026-01-09,3000182,102484482,"""thiphuonghoa.hoang@concentrix.…","""2100-0600""","""Master_Schedule_Merged.xlsx"""
…,…,…,…,…,…
2026-05-13,3112809,103305150,"""thithuyduong.duong@concentrix.…","""2200-0700""","""Master_Schedule_Merged.xlsx"""
2026-05-14,3112809,103305150,"""thithuyduong.duong@concentrix.…","""OFF""","""Master_Schedule_Merged.xlsx"""
2026-05-15,3112809,103305150,"""thithuyduong.duong@concentrix.…","""OFF""","""Master_Schedule_Merged.xlsx"""


In [23]:
# ==================================================================
# ===== CELL 5: ETL (FIX LOGIC: HAL UNPLANNED = 0.5) =====
# ==================================================================
# ==================================================================
# ===== 1. LOADING FUNCTIONS =====
# ==================================================================

def _load_single_rta_file(f: Path) -> pl.DataFrame:
    """Load one RTA file; used by parallel executor."""
    if f.suffix == '.xlsx':
        return pl.read_excel(f, engine='calamine', infer_schema_length=0)
    return pl.read_csv(f, infer_schema_length=0)


def load_rta_report(input_folder: Path) -> pl.DataFrame:
    print(f"Loading RTA Reports from: {input_folder}")

    files = list(input_folder.glob("*.xlsx")) + list(input_folder.glob("*.csv"))

    # OPTIMIZATION: parallel IO with ThreadPoolExecutor
    dfs: list[pl.DataFrame] = []
    with ThreadPoolExecutor() as ex:
        futures = {ex.submit(_load_single_rta_file, f): f for f in files}
        for fut in as_completed(futures):
            try:
                dfs.append(fut.result())
            except Exception as e:
                print(f"  Warning: could not load {futures[fut].name}: {e}")

    df = pl.concat(dfs, how="vertical_relaxed")

    output_path = folder_paths["resources"] / "_rta_report_combination.parquet"
    df.write_parquet(output_path)

    return df


def _load_single_hc_file(p: Path) -> pl.DataFrame | None:
    """Load one HC Extend file; used by parallel executor."""
    ext = p.suffix.lower()
    if ext not in {".xlsx", ".csv"} or p.name.startswith("~$"):
        return None
    if ext == ".xlsx":
        df = pl.read_excel(p, sheet_name=None, infer_schema_length=0)
    else:
        try:
            df = pl.read_csv(p, encoding="utf-8", infer_schema_length=0)
        except Exception:
            df = pl.read_csv(p, encoding="ISO-8859-1", ignore_errors=True, infer_schema_length=0)
    return df.with_columns(pl.all().cast(pl.String))


def load_hc_extend(input_folder: Path) -> pl.DataFrame:
    print(f"Loading HC Extend from: {input_folder}")
    all_paths = [p for p in input_folder.glob("*.*")]

    # OPTIMIZATION: parallel IO with ThreadPoolExecutor
    dfs: list[pl.DataFrame] = []
    files_found = 0
    with ThreadPoolExecutor() as ex:
        futures = {ex.submit(_load_single_hc_file, p): p for p in all_paths}
        for fut in as_completed(futures):
            result = fut.result()
            if result is not None:
                files_found += 1
                dfs.append(result)

    if files_found == 0:
        return pl.DataFrame()
    combined_df = pl.concat(dfs, how="vertical_relaxed") if dfs else pl.DataFrame()
    output_path = folder_paths["resources"] / "hc_extend_combination.parquet"
    combined_df.write_parquet(output_path)
    return combined_df


# ==================================================================
# ===== 2. HELPER FUNCTIONS =====
# ==================================================================

def clean_dataframe_keys(df: LazyFrame, source_type: str) -> LazyFrame:
    # OPTIMIZATION: call collect_schema() once and cache
    schema = df.collect_schema()
    schema_dict = dict(zip(schema.names(), schema.dtypes()))

    date_col = "Date"
    if source_type == "roster":
        date_col = "Scheduled_Date"
        if "Date" in schema_dict and "Scheduled_Date" not in schema_dict:
            df = df.rename({"Date": "Scheduled_Date"})
            schema_dict["Scheduled_Date"] = schema_dict.pop("Date")

    if date_col in schema_dict:
        current_dtype = schema_dict[date_col]
        if current_dtype == pl.String:
            df = df.with_columns(pl.col(date_col).str.to_datetime(strict=False).dt.date().alias(date_col))
        elif current_dtype in (pl.Date, pl.Datetime):
            df = df.with_columns(pl.col(date_col).dt.date().alias(date_col))

    if source_type == "rta":
        if "Date_Converted" not in schema_dict and "Date" in schema_dict:
            df = df.with_columns(pl.col("Date").alias("Date_Converted"))
            schema_dict["Date_Converted"] = schema_dict.get("Date")

        if "Date_Converted" in schema_dict:
            current_dtype_rta = schema_dict["Date_Converted"]
            if current_dtype_rta == pl.String:
                df = df.with_columns(pl.col("Date_Converted").str.to_datetime(strict=False).dt.date().alias("Date_Converted"))
            elif current_dtype_rta in (pl.Date, pl.Datetime):
                df = df.with_columns(pl.col("Date_Converted").dt.date().alias("Date_Converted"))

    if "Email" in schema_dict and "Email Id" not in schema_dict:
        df = df.with_columns(pl.col("Email").alias("Email Id"))
        schema_dict["Email Id"] = schema_dict.pop("Email")

    if "Email Id" in schema_dict:
        if schema_dict["Email Id"] == pl.String:
            df = df.with_columns(pl.col("Email Id").str.strip_chars().str.to_lowercase().alias("Email Id"))

    iex_col = "IEX ID"
    if source_type == "leave": iex_col = "IEX"
    if iex_col in schema_dict: df = df.with_columns(pl.col(iex_col).cast(pl.Float64, strict=False).alias(iex_col))
    if "OracleID" in schema_dict: df = df.with_columns(pl.col("OracleID").cast(pl.Float64, strict=False).alias("OracleID"))
    return df


def parse_time_standalone(x) -> time | None:
    if pd.isna(x) or x == "": return None
    x = str(x).strip()
    try:
        if len(x) == 4: return pd.to_datetime(x, format="%H%M", errors="raise").time()
        if len(x) == 5: return pd.to_datetime(x, format="%H:%M", errors="raise").time()
        return pd.to_datetime(x, errors="raise").time()
    except Exception:
        val = pd.to_datetime(x, errors="coerce")
        if pd.isna(val): return None
        return val.time()


def parse_time_expr(str_expr: pl.Expr) -> pl.Expr:
    # OPTIMIZATION: try Polars native time parsing first; fall back to map_elements only if needed
    return pl.coalesce(
        str_expr.str.to_time("%H:%M", strict=False),
        str_expr.str.to_time("%H%M", strict=False),
        str_expr.str.to_time("%H:%M:%S", strict=False),
        str_expr.map_elements(parse_time_standalone, return_dtype=pl.Time),
    )


def polars_split_shift(shift_col_name: str) -> list[pl.Expr]:
    split_expr = pl.col(shift_col_name).str.strip_chars().str.split_exact(by="-", n=1)
    start_str_expr = split_expr.struct[0].str.strip_chars()
    end_str_expr = split_expr.struct[1].str.strip_chars()
    return [parse_time_expr(start_str_expr), parse_time_expr(end_str_expr)]


def pick_col(df: pl.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns: return c
    return None


def sum_and_format_time_expr(col: str, alias: str, positive_only: bool = False, threshold_sec: int = 180) -> pl.Expr:
    # OPTIMIZATION: removed .unique() which was incorrect semantics for aggregation
    # (unique deduplicates values inside each group which is wrong for a sum)
    base_expr = pl.col(col).cast(pl.Float64, strict=False).drop_nulls()
    if positive_only:
        filtered_expr = base_expr.filter(base_expr > 0)
    else:
        filtered_expr = base_expr.filter(base_expr > (threshold_sec / 3600.0))
    return filtered_expr.sum().alias(alias)


def smart_to_hours_expr(col: str) -> pl.Expr:
    float_val = pl.col(col).cast(pl.Float64, strict=False)
    time_val = pl.col(col).str.to_time(strict=False)
    time_to_hours = (
        time_val.dt.hour() +
        (time_val.dt.minute() / 60.0) +
        (time_val.dt.second() / 3600.0)
    )
    return pl.coalesce(float_val, time_to_hours).fill_null(0.0)


# ==================================================================
# ===== 3. MAIN ETL BUILD FUNCTIONS =====
# ==================================================================

def build_roster_prepared(roster_raw: pl.DataFrame) -> pl.DataFrame:
    print("      Processing Roster...")
    lf = roster_raw.lazy()
    lf = (
        lf.sort(pl.col("Shift") == "Termination", descending=True)
          .unique(subset=["IEX ID", "Scheduled_Date"], keep="first")
    )
    lf = lf.rename({"Scheduled_Date": "Date"})
    start_expr, end_expr = polars_split_shift("Shift")
    lf = lf.with_columns(start_expr.alias("Start_Shift"), end_expr.alias("End_Shift"))
    lf = lf.with_columns(
        pl.datetime(pl.col("Date").dt.year(), pl.col("Date").dt.month(), pl.col("Date").dt.day(),
                    pl.col("Start_Shift").dt.hour(), pl.col("Start_Shift").dt.minute(), pl.col("Start_Shift").dt.second()).alias("Datetime_Start_Shift"),
        pl.datetime(pl.col("Date").dt.year(), pl.col("Date").dt.month(), pl.col("Date").dt.day(),
                    pl.col("End_Shift").dt.hour(), pl.col("End_Shift").dt.minute(), pl.col("End_Shift").dt.second()).alias("Datetime_End_Shift")
    ).rename({"Date": "Scheduled_Date"})
    night_shift_mask = (pl.col("Datetime_End_Shift").is_not_null() & pl.col("Datetime_Start_Shift").is_not_null() &
                        (pl.col("Datetime_End_Shift") <= pl.col("Datetime_Start_Shift")))
    lf = lf.with_columns(
        pl.when(night_shift_mask).then(pl.col("Datetime_End_Shift") + pl.duration(days=1))
          .otherwise(pl.col("Datetime_End_Shift")).alias("Datetime_End_Shift"),
        pl.when(night_shift_mask).then(1).otherwise(0).alias("Night_Shift")
    )
    cols = ["Scheduled_Date", "Email Id", "IEX ID", "OracleID", "Shift", "Start_Shift", "End_Shift",
            "Night_Shift", "Datetime_Start_Shift", "Datetime_End_Shift"]
    # OPTIMIZATION: cache schema once instead of calling collect_schema() in list comprehension
    schema_names = set(lf.collect_schema().names())
    final_cols = [c for c in cols if c in schema_names]
    return lf.select(final_cols).collect()


def build_hc_extend_prepared(hc_raw: pl.DataFrame, roster_prepared: pl.DataFrame) -> pl.DataFrame:
    print("      Processing HC_Extend...")
    hc_ext = hc_raw.lazy().join(roster_prepared.lazy(), how="left", left_on=["Date", "Email Id"], right_on=["Scheduled_Date", "Email Id"]).collect()
    keep = ["Year", "Month", "Week Begin", "Date", "OracleID", "IEX ID", "Employee Name", "Alias", "LOB", "Supervisor Name", "Email Id", "Wave",
            "Shift", "Start_Shift", "End_Shift", "Night_Shift", "Datetime_Start_Shift", "Datetime_End_Shift", "Site", "Day", "TL ID", "Supervisor Email", "Manager/OM Name", "LWD/Movement", "Detail Status", "LOB_2", "LOB_3"]
    hc_cols = set(hc_ext.columns)
    return hc_ext.select([c for c in keep if c in hc_cols])


def build_leave_prepared(leave_raw: pl.DataFrame) -> pl.DataFrame:
    print("      Processing Leave...")
    lf = leave_raw.lazy().unique(subset=["Date", "IEX"], keep="last")

    fs_start, fs_end = polars_split_shift("Final Shift")
    s_start, s_end = polars_split_shift("Shift")
    lf = lf.with_columns(fs_start.alias("Final_Start_Shift"), fs_end.alias("Final_End_Shift"),
                         s_start.alias("Start_Shift"), s_end.alias("End_Shift"))

    date2_mask = (pl.col("Final_Start_Shift").is_not_null() & pl.col("Start_Shift").is_not_null() & (pl.col("Final_Start_Shift") < pl.col("Start_Shift")))
    lf = lf.with_columns(pl.when(date2_mask).then(pl.col("Date") + pl.duration(days=1)).otherwise(None).alias("Date2"))

    def make_dt(d, t): return pl.datetime(d.dt.year(), d.dt.month(), d.dt.day(), t.dt.hour(), t.dt.minute(), t.dt.second())

    lf = lf.with_columns(
        pl.when(pl.col("Date2").is_not_null()).then(make_dt(pl.col("Date2"), pl.col("Final_Start_Shift")))
          .otherwise(make_dt(pl.col("Date"), pl.col("Final_Start_Shift"))).alias("Leave_Datetime_Start_Shift")
    )

    end_logic = (pl.when(pl.col("Final_End_Shift").is_null()).then(None)
        .when(pl.col("Date2").is_not_null()).then(make_dt(pl.col("Date2"), pl.col("Final_End_Shift")))
        .when(pl.col("Final_Start_Shift").is_not_null()).then(
            pl.when(pl.col("Final_End_Shift") < pl.col("Final_Start_Shift"))
              .then(make_dt(pl.col("Date"), pl.col("Final_End_Shift")) + pl.duration(days=1))
              .otherwise(make_dt(pl.col("Date"), pl.col("Final_End_Shift")))
        ).otherwise(make_dt(pl.col("Date"), pl.col("Final_End_Shift"))))

    return lf.with_columns(end_logic.alias("Leave_Datetime_End_Shift")).drop("Date2").collect()


# --- BUILD ACTIVITY ---
def build_activity_prepared(rta: pl.DataFrame) -> pl.DataFrame:
    print("      Processing RTA (DAX simulation)...")
    r = rta
    if "Date_Converted" not in r.columns: r = r.with_columns(pl.lit(None).cast(pl.Datetime).alias("Date_Converted"))
    r = r.with_columns(
        pl.col("Date_Converted").dt.date().alias("Date"),
        pl.col("Date_Converted").dt.year().alias("Year"),
        pl.col("Date_Converted").dt.strftime("%b-%y").alias("Month"),
        pl.col("Date_Converted").dt.truncate("1w").alias("Week Begin")
    )

    src = {
        "Shift Tracking":     pick_col(r, ["Shift Tracking", "shift_tracking_iex", "shift_tracking"]),
        "Roster Presented":   pick_col(r, ["Roster Presented", "roster_presented_iex", "roster_presented"]),
        "Roster Scheduled":   pick_col(r, ["Roster Scheduled", "roster_scheduled_iex", "roster_scheduled"]),
        "Datetime_First_Start_Shift": pick_col(r, ["Datetime_First_Start_Shift", "_start_shift_iex"]),
        "Datetime_First_End_Shift":   pick_col(r, ["Datetime_First_End_Shift", "_end_shift_iex"]),
        "First Shift":        pick_col(r, ["First Shift", "first_shift_iex", "first_shift"]),
        "Fluctuate Shift":    pick_col(r, ["Fluctuate Shift", "fluctuate_shift_iex", "fluctuate_shift", "Shift"]),
        "Night_Shift":        pick_col(r, ["Night_Shift", "_night_shift_iex", "night_shift", "Night Shift"]),
        "Target":             pick_col(r, ["Target", "_target_iex", "target"]),
        "start":              pick_col(r, ["start", "_start_time"]),
        "end":                pick_col(r, ["end", "_end_time"]),
        "duration":           pick_col(r, ["duration", "_duration"]),
        "hc_actual":          pick_col(r, ["hc_actual", "_hc_actual"]),
        "lateness":           pick_col(r, ["lateness", "_lateness"]),
        "Open Time":          pick_col(r, ["Open Time", "_open_time_iex", "open_time"]),
        "Extra Time":         pick_col(r, ["Extra Time", "_extra_time_iex", "extra_time", "ot"]),
        "training":           pick_col(r, ["Training", "training", "_training_iex", "Training Hours"]),
        "ncns":               pick_col(r, ["NCNS", "ncns", "_ncns_iex", "NCNS Duration"]),
        "sum_productive":     pick_col(r, ["sum_productive", "_productive", "SUM Productive"]),
        "nsa_group":          pick_col(r, ["nsa_group", "_nsa_group_ramco"]),
        "ot_group":           pick_col(r, ["ot_group", "_ot_group_ramco"]),
        "nm_group":           pick_col(r, ["nm_group", "_nm_group_ramco"]),
        "ramco_marked":       pick_col(r, ["ramco_marked", "_ramco_marked_ramco"]),
        "time_late":          pick_col(r, ["time_late", "_late"]),
        "time_leave":         pick_col(r, ["time_leave", "_leave"]),
        "training-idle":      pick_col(r, ["training-idle", "_training_idle"]),
        "coaching-idle":      pick_col(r, ["coaching-idle", "_coaching_idle"]),
        "orther_status":      pick_col(r, ["other_status", "orther_status", "_orther_status"]),
        "lunch":              pick_col(r, ["lunch", "_lunch"]),
        "break":              pick_col(r, ["break", "_break"]),
        "over_lunch":         pick_col(r, ["over_lunch", "_over_lunch"]),
        "over_break":         pick_col(r, ["over_break", "_over_break"]),
        "missed_productive":  pick_col(r, ["missed_productive", "_missed_productive"]),
    }

    src = {k: v for k, v in src.items() if v}
    src_inv = {v: k for k, v in src.items()}
    group_by_cols = ["Date", "Email Id"]
    base_cols = ["Date", "Email Id", "Year", "Month", "Week Begin", "OracleID", "IEX ID", "Employee Name", "Alias", "LOB", "Supervisor Name", "Wave"]
    for c in base_cols:
        if c not in r.columns: r = r.with_columns(pl.lit(None).alias(c))

    aggs = [
        pl.col("Year").max(), pl.col("Month").max(), pl.col("Week Begin").max(),
        pl.col("OracleID").max(), pl.col("IEX ID").max(),
        pl.col("Employee Name").drop_nulls().first(), pl.col("Alias").drop_nulls().first(),
        pl.col("LOB").drop_nulls().first(), pl.col("Supervisor Name").drop_nulls().first(), pl.col("Wave").drop_nulls().first(),
    ]

    # String columns (NSA Group, Ramco Marked, etc.)
    for k in ["Shift Tracking", "First Shift", "Fluctuate Shift", "nsa_group", "ot_group", "nm_group", "ramco_marked"]:
        if k in src: aggs.append(pl.col(src[k]).drop_nulls().first().alias(src[k]))

    # Numeric columns: cast to float before sum
    sum_cols_numeric = [
        "Roster Presented", "Roster Scheduled", "Night_Shift", "Target", "duration", "hc_actual",
        "lateness", "Open Time", "Extra Time", "sum_productive",
        "training", "ncns",
        "training-idle", "coaching-idle", "break", "orther_status"
    ]

    for k in sum_cols_numeric:
        if k in src:
            aggs.append(pl.col(src[k]).cast(pl.Float64, strict=False).fill_null(0.0).sum().alias(src[k]))

    # Start/End datetime columns: use str.to_datetime (native, no map_elements)
    for k in ["start", "end", "Datetime_First_Start_Shift", "Datetime_First_End_Shift"]:
        if k in src: aggs.append(pl.col(src[k]).str.to_datetime(strict=False).max().alias(src[k]))

    thr_sec = 180
    if "time_late" in src: aggs.append(sum_and_format_time_expr(src["time_late"], "Measure_Time_Late", False, thr_sec))
    if "time_leave" in src: aggs.append(sum_and_format_time_expr(src["time_leave"], "Measure_Time_Leave", False, thr_sec))
    if "training-idle" in src: aggs.append(sum_and_format_time_expr(src["training-idle"], "Measure_Training_Idle", False, thr_sec))
    if "coaching-idle" in src: aggs.append(sum_and_format_time_expr(src["coaching-idle"], "Measure_Coaching_Idle", False, thr_sec))
    if "orther_status" in src: aggs.append(sum_and_format_time_expr(src["orther_status"], "Measure_Orther_Status", False, thr_sec))
    if "lunch" in src: aggs.append(sum_and_format_time_expr(src["lunch"], "Measure_Lunch", True))
    if "break" in src: aggs.append(sum_and_format_time_expr(src["break"], "Measure_Break", True))
    if "over_lunch" in src: aggs.append(sum_and_format_time_expr(src["over_lunch"], "Measure_Over_Lunch", True))
    if "over_break" in src: aggs.append(sum_and_format_time_expr(src["over_break"], "Measure_Over_Break", True))
    if "over_lunch" in src: aggs.append((pl.col(src["over_lunch"]).cast(pl.Float64, strict=False) > 0).sum().alias("Measure_over_lunch_count"))
    if "over_break" in src: aggs.append((pl.col(src["over_break"]).cast(pl.Float64, strict=False) > 0).sum().alias("Measure_over_break_count"))

    lf = r.lazy().group_by(group_by_cols).agg(aggs)
    print(f"            GroupBy (Date, Email Id) created...")

    # OPTIMIZATION: call collect_schema() once and reuse
    lf_schema_names = set(lf.collect_schema().names())

    def get_agg_col(key: str) -> pl.Expr:
        col_name_val = src.get(key)
        if col_name_val and col_name_val in lf_schema_names:
            return pl.col(col_name_val).cast(pl.Float64, strict=False).fill_null(0.0)
        return pl.lit(0.0)

    Target = get_agg_col("Target")
    Productive = get_agg_col("sum_productive")
    Training_Sec = get_agg_col("training-idle")
    Coaching_Sec = get_agg_col("coaching-idle")
    Break_Sec = get_agg_col("break")
    Other_Sec = get_agg_col("orther_status")

    Unproductive = Training_Sec + Coaching_Sec + Break_Sec + Other_Sec
    Total = Unproductive + Productive
    Result = pl.when(Total == 0).then(None).otherwise(Unproductive / Total)  # noqa: F841

    Total_Conf = Training_Sec + Coaching_Sec + Productive
    Result_Conf = pl.when(Target == 0).then(None).otherwise(Total_Conf / Target)
    lf = lf.with_columns(
        pl.when((Target == 0) | (Result_Conf <= 0)).then(None).otherwise(Result_Conf).alias("Measure_Confirmance")
    )

    missed_val = Target - (Productive / 3600.0)
    lf = lf.with_columns(pl.max_horizontal(0.0, missed_val).alias("Measure_missed_hours"))

    print("            All DAX Measures computed.")

    rename_map = {
        "Datetime_First_Start_Shift": "Datetime_Start_Shift", "Datetime_First_End_Shift": "Datetime_End_Shift",
        "Fluctuate Shift": "Shift", "duration": "Duration", "hc_actual": "HC Actual",
        "Measure_Time_Late": "Time_Late", "Measure_Time_Leave": "Time_Leave",
        "lateness": "Lateness", "Extra Time": "OT", "training": "Training", "ncns": "NCNS", "sum_productive": "SUM Productive",
        "nsa_group": "NSA Group", "ot_group": "OT Group", "nm_group": "NM Group", "ramco_marked": "Ramco Marked",
        "Measure_Training_Idle": "Training_Idle [Sum]", "Measure_Coaching_Idle": "Coaching_Idle [Sum]",
        "Measure_Orther_Status": "Orther_Status [Sum]", "Measure_Lunch": "Lunch [Sum]", "Measure_Break": "Break [Sum]",
        "start": "Start Time", "end": "End Time", "Alias": "Alias Name"
    }

    # OPTIMIZATION: compute final_rename using the already-cached schema name set
    final_rename = {}
    for col in lf_schema_names:
        original_name = src_inv.get(col)
        if original_name and original_name in rename_map: final_rename[col] = rename_map[original_name]
        elif col.startswith("Measure_") and col in rename_map: final_rename[col] = rename_map[col]
    if "Alias" in lf_schema_names: final_rename["Alias"] = "Alias Name"

    agg = lf.rename(final_rename).collect()
    print("      Activity columns renamed.")
    return agg


# --- BUILD ATD (FIX PRESENT, PLANNED, UNPLANNED & BLANK COLUMNS) ---
def build_atd_final(hc_ext: pl.DataFrame, activity_df: pl.DataFrame, leave_prep: pl.DataFrame) -> pl.DataFrame:
    print("      Starting ATD computation (M code 'ATD_Final' simulation)...")

    # 1. Base join
    if "Date" not in activity_df.columns: activity_df = activity_df.with_columns(pl.lit(None).cast(pl.Date).alias("Date"))
    if "Email Id" not in activity_df.columns: activity_df = activity_df.with_columns(pl.lit(None).cast(pl.String).alias("Email Id"))

    base_lf = hc_ext.lazy().join(
        activity_df.lazy(),
        how="left",
        on=["Date", "Email Id"],
        suffix="_Activity"
    )

    # OPTIMIZATION: cache schema once and reuse throughout the function
    cols = set(base_lf.collect_schema().names())

    # 2. Night Shift Priority logic
    ns_expr = pl.lit(None)
    if "Night_Shift_Activity" in cols and "Night_Shift" in cols: ns_expr = pl.coalesce(pl.col("Night_Shift_Activity"), pl.col("Night_Shift"))
    elif "Night_Shift_Activity" in cols: ns_expr = pl.col("Night_Shift_Activity")
    elif "Night_Shift" in cols: ns_expr = pl.col("Night_Shift")
    base_lf = base_lf.with_columns(ns_expr.alias("Final_Night_Shift_Priority"))

    # 3. Coalesce duplicated columns from join suffix
    coalesce_targets = [
        "Year", "Month", "Week Begin", "OracleID", "IEX ID", "Employee Name", "Alias Name", "LOB", "Supervisor Name", "Wave", "Target", "HC Actual"
    ]
    for col in coalesce_targets:
        col_act = f"{col}_Activity"
        if col_act in cols:
            if col == "Alias Name" and "Alias" in cols:
                base_lf = base_lf.with_columns(pl.coalesce(pl.col("Alias"), pl.col(col_act)).alias(col)).drop([col_act, "Alias"])
            elif col == "Alias Name": base_lf = base_lf.rename({col_act: col})
            elif col in cols: base_lf = base_lf.with_columns(pl.coalesce(pl.col(col), pl.col(col_act)).alias(col)).drop(col_act)
            else: base_lf = base_lf.rename({col_act: col})
            # Update cached schema after each mutation
            cols = set(base_lf.collect_schema().names())

    base_lf = base_lf.rename({
        "Shift": "Roster.Shift", "Start_Shift": "Roster.Start_Shift", "End_Shift": "Roster.End_Shift",
        "Night_Shift": "Roster.Night_Shift", "Datetime_Start_Shift": "Roster.Datetime_Start_Shift", "Datetime_End_Shift": "Roster.Datetime_End_Shift"
    })
    base_lf = base_lf.rename({"Final_Night_Shift_Priority": "Night_Shift"})

    # HC SCHEDULE
    rta_shift = pl.col("Roster.Shift")
    up_act = rta_shift.str.to_uppercase()
    shift_hours = (pl.col("Roster.Datetime_End_Shift") - pl.col("Roster.Datetime_Start_Shift")).dt.total_seconds() / 3600.0

    base_lf = base_lf.with_columns(
        pl.when(up_act.str.contains("HAL|HAB|HLWP|HSL")).then(1.0)
          .when(rta_shift.is_null()).then(None)
          .when((rta_shift.str.contains("-") & (shift_hours > 6)) | (up_act.is_in({"AL", "CO", "LWP", "AB", "SL", "CMLF"}))).then(1.0)
          .when(rta_shift.str.contains("-") & (shift_hours < 6)).then(0.5)
          .otherwise(0.0).alias("HC Schedule")
    )

    print("            Joining Leave...")
    leave_ren = leave_prep.lazy().select(
        pl.col("Date"), pl.col("IEX"), pl.col("Final Shift").alias("Leave.Shift"),
        pl.col("Leave_Datetime_Start_Shift").alias("Leave.Leave_Datetime_Start_Shift"),
        pl.col("Leave_Datetime_End_Shift").alias("Leave.Leave_Datetime_End_Shift"),
        pl.col("Leave"), pl.col("Reason"), pl.col("Remark").alias("Leave.Remark")
    )
    base_lf = base_lf.join(leave_ren, how="left", left_on=["Date", "IEX ID"], right_on=["Date", "IEX"])
    base_lf = base_lf.with_columns(
        pl.col("Leave.Remark").alias("Remark"),
        pl.coalesce(pl.col("Leave.Leave_Datetime_Start_Shift"), pl.col("Roster.Datetime_Start_Shift")).alias("Datetime_Start_Shift"),
        pl.coalesce(pl.col("Leave.Leave_Datetime_End_Shift"), pl.col("Roster.Datetime_End_Shift")).alias("Datetime_End_Shift"),
        pl.coalesce(pl.col("Leave.Shift"), pl.col("Roster.Shift")).alias("Original.Shift")
    ).filter(pl.col("Original.Shift").is_not_null())

    print("            Computing Present, Planned, Unplanned...")
    s_eff = pl.col("Original.Shift").str.to_uppercase()
    dash = pl.col("Original.Shift").str.contains("-")
    hrs = (pl.col("Datetime_End_Shift") - pl.col("Datetime_Start_Shift")).dt.total_seconds() / 3600.0

    # OPTIMIZATION: check schema once instead of calling collect_schema() inside conditional
    current_cols = set(base_lf.collect_schema().names())
    if "Start Time" not in current_cols:
        base_lf = base_lf.with_columns(pl.lit(None).alias("Start Time"))

    # --- ATTENDANCE (with fallback) ---
    has_activity = (pl.col("Start Time").is_not_null()) | (pl.col("Duration").fill_null(0) > 0) | (pl.col("Target").fill_null(0) > 0)

    att = (pl.when(s_eff.is_in({"AL", "LWP", "CMLF", "AB", "SL", "CO"})).then(s_eff)
           .when(pl.col("Leave").is_not_null()).then(pl.col("Leave"))
           .when(s_eff == "OFF").then(pl.lit("WO")).when(s_eff == "HO").then(pl.lit("HO"))
           .when(pl.col("Leave").is_null() & has_activity & dash).then(pl.lit("PR"))
           .when(pl.col("Leave").is_null() & ~has_activity).then(pl.lit("NCNS")).otherwise(None))
    base_lf = base_lf.with_columns(att.alias("Attendance"))

    # --- PRESENT (HAL -> 0.5) ---
    pres = (pl.when(dash & (pl.col("Attendance") == "PR") & (hrs < 6)).then(0.5)
            .when(s_eff.str.contains("HAL") & (pl.col("Attendance") == "PR") & (hrs > 6)).then(0.5)
            .when(dash & (pl.col("Attendance") == "PR") & (hrs > 6)).then(1.0)
            # HAL/HAB/HSL -> 0.5
            .when(dash & pl.col("Attendance").is_in(["HAL", "HAB", "HSL", "HLWP"])).then(0.5)
            .when(dash & s_eff.str.contains("HAL|HAB|HSL|HLWP")).then(0.5).otherwise(0.0))
    base_lf = base_lf.with_columns(pres.alias("Present"))

    # --- PLANNED (original logic, no HAL=0.5 override) ---
    planned = (pl.when(s_eff.str.contains("HAL|HLWP|HAB|HSL")).then(0.5)
               .when(s_eff.is_in({"AL", "CO", "AB", "SL", "LWP", "CMLF"})).then(1.0)
               .otherwise(0.0))
    base_lf = base_lf.with_columns(planned.alias("Planned"))

    unp = pl.col("HC Schedule").fill_null(0) - pl.col("Present") - pl.col("Planned")
    base_lf = base_lf.with_columns(pl.when(unp < 0).then(0.0).otherwise(unp).alias("Unplanned"))

    # --- FIX BLANK COLUMNS: batch with_columns calls ---
    final_cols_schema = set(base_lf.collect_schema().names())

    numeric_fix = [
        pl.col(c).cast(pl.Float64, strict=False) if c in final_cols_schema
        else pl.lit(None).cast(pl.Float64).alias(c)
        for c in ["SUM Productive", "Open Time", "Target", "HC Actual"]
    ]
    base_lf = base_lf.with_columns(numeric_fix)

    for c in ["NSA Group", "OT Group", "NM Group", "Ramco Marked"]:
        c_act = f"{c}_Activity"
        if c_act in final_cols_schema:
            base_lf = base_lf.rename({c_act: c}).with_columns(pl.col(c).cast(pl.String))
        elif c in final_cols_schema:
            base_lf = base_lf.with_columns(pl.col(c).cast(pl.String))
        else:
            base_lf = base_lf.with_columns(pl.lit(None).cast(pl.String).alias(c))

    base_lf = base_lf.with_columns(
        pl.when(pl.col("Open Time").fill_null(0) != 0)
          .then((pl.col("SUM Productive").fill_null(0) / 3600.0) / pl.col("Open Time")).otherwise(None).alias("In Office"),
        ((pl.col("HC Actual") == pl.col("Present")) | (pl.col("HC Actual").is_null() & (pl.col("Present") == 0))).alias("Re_Check")
    )
    rm = pl.col("Ramco Marked").cast(pl.String)
    rm_chk = ((pl.col("Attendance") == rm) | (pl.col("Attendance") == "PR") & (rm.is_in({"PO", "PH"})) |
              (pl.col("Attendance").is_null() & (rm == "HO")) | (~pl.col("Original.Shift").str.contains("-") & rm.is_not_null()))
    base_lf = base_lf.with_columns(rm_chk.alias("Ramco_ReCheck"))
    hc, pr, pd_sum = pl.col("HC Actual"), pl.col("Present"), pl.col("SUM Productive").fill_null(0)
    final_hc = (pl.when(hc.is_null()).then(None).when((hc <= pr) & (hc != 0)).then(pr)
                .when(hc > pr).then(
                    pl.when(pr == 0.5).then(pl.when(pd_sum >= 3).then(pr).otherwise(0.0))
                    .when(pr == 1.0).then(pl.when((pd_sum < 3) & (pd_sum >= 0)).then(0.0).otherwise(hc))
                    .when(pr == 0.0).then(pl.when((pd_sum < 3) & (pd_sum >= 0)).then(0.0).otherwise(hc)).otherwise(hc)
                ).otherwise(0.0))
    base_lf = base_lf.with_columns(final_hc.alias("Final_HC_Actual"), pl.col("HC Schedule").sum().over("Date").alias("HC_Schedule_Sum"))
    print("            Join complete. Deduplicating...")
    return base_lf.collect().unique()


def shape_for_report(df: pl.DataFrame) -> pl.DataFrame:
    print("      Shaping columns for output...")
    cols = [
        "Year", "Month", "Week Begin", "Date", "OracleID", "IEX ID", "Employee Name", "Alias Name", "LOB",
        "Supervisor Name", "Email Id", "Wave",
        "Shift Tracking", "Roster Presented", "Roster Scheduled",
        "Datetime_Start_Shift", "Datetime_End_Shift", "First Shift", "Original.Shift", "Night_Shift",
        "Target", "Start Time", "End Time", "Duration", "HC Actual",
        "Time_Late", "Time_Leave", "Lateness", "Open Time", "OT", "Training", "NCNS", "SUM Productive",
        "Training_Idle [Sum]", "Coaching_Idle [Sum]", "Orther_Status [Sum]", "Lunch [Sum]", "Break [Sum]",
        "Measure_Confirmance", "Measure_over_lunch_count", "Measure_over_break_count",
        "Measure_missed_hours", "Measure_Over_Lunch", "Measure_Over_Break",
        "Attendance", "Reason", "Remark", "Present", "Planned", "Unplanned",
        "HC Schedule", "HC_Schedule_Sum", "In Office",
        "NSA Group", "OT Group", "NM Group", "Ramco Marked", "Re_Check", "Ramco_ReCheck", "Final_HC_Actual"
    ]
    df_cols = set(df.columns)
    return df.select([pl.col(c) if c in df_cols else pl.lit(None).alias(c) for c in cols])


print("All ETL functions defined (Optimized Version).")

All ETL functions defined (Optimized Version).


In [24]:
# ==================================================================
# ===== CELL 6: MAIN ETL EXECUTION BLOCK =====
# ==================================================================

print("--- STARTING AUTOMATED ETL PIPELINE ---")

try:
    # Step 1: Load raw data (Leave & Roster were run in Cell 3 & 4)
    print("\n--- STEP 1: LOAD RAW DATA ---")

    # Load RTA and HC in parallel using threads
    with ThreadPoolExecutor(max_workers=2) as ex:
        fut_rta = ex.submit(load_rta_report, folder_paths["rta_output"])
        fut_hc  = ex.submit(load_hc_extend, folder_paths["hc_extend"])
        rta_raw = fut_rta.result()
        hc_raw  = fut_hc.result()

    # Get Leave & Roster from variables set in Cell 3 & 4
    if 'LEAVE_ADD_DAT' not in locals():
        raise NameError("'LEAVE_ADD_DAT' not found. Please run Cell 3.")
    if 'combined' not in locals():
        raise NameError("'combined' (Roster) not found. Please run Cell 4.")

    leave_raw = LEAVE_ADD_DAT
    roster_raw = combined  # 'combined' is the Roster DataFrame from Cell 4

    print(f"      hc_raw (raw): {len(hc_raw)} rows")
    print(f"      rta_raw (raw): {len(rta_raw)} rows")
    print(f"      roster_raw (from Cell 4): {len(roster_raw)} rows")
    print(f"      leave_raw (from Cell 3): {len(leave_raw)} rows")

    # Guard: HC must not be empty
    if len(hc_raw) == 0:
        print("\n*** CRITICAL ERROR: `hc_raw` is empty. ***")
        print(f"*** Please check path: {folder_paths['hc_extend']} ***")
        raise FileNotFoundError("HC data not found — pipeline cannot continue.")

    # Step 2: Normalize dtypes (lazy, all 4 sources in parallel)
    print("\n--- STEP 2: NORMALIZE KEY DTYPES (LAZY PARALLEL) ---")
    with ThreadPoolExecutor(max_workers=4) as ex:
        fut_hc_c  = ex.submit(lambda: clean_dataframe_keys(hc_raw.lazy(),     "hc").collect())
        fut_ros_c = ex.submit(lambda: clean_dataframe_keys(roster_raw.lazy(), "roster").collect())
        fut_lv_c  = ex.submit(lambda: clean_dataframe_keys(leave_raw.lazy(),  "leave").collect())
        fut_rta_c = ex.submit(lambda: clean_dataframe_keys(rta_raw.lazy(),    "rta").collect())
        hc_raw     = fut_hc_c.result()
        roster_raw = fut_ros_c.result()
        leave_raw  = fut_lv_c.result()
        rta_raw    = fut_rta_c.result()
    print("      Key dtypes normalized.")

    # Steps 3-6: Run full ETL pipeline
    print("\n--- STEPS 3-6: RUN ETL PIPELINE ---")
    roster_p = build_roster_prepared(roster_raw)
    hc_ext   = build_hc_extend_prepared(hc_raw, roster_p)
    leave_p  = build_leave_prepared(leave_raw)

    print("\n--- STEP 4: ACTIVITY PROCESSING (DAX + RENAME) ---")
    activity_df_final = build_activity_prepared(rta_raw)

    print("\n--- STEP 5: ATD_FINAL COMPUTATION ---")
    atd_raw = build_atd_final(hc_ext, activity_df_final, leave_p)

    print("\n--- STEP 6: SHAPE AND EXPORT ---")
    atd_out = shape_for_report(atd_raw)  # Final Polars DataFrame

    # Save outputs

    # # --- Save as Excel (original approach, commented out) ---
    # output_path_excel = folder_paths["resources"] / "ATD_Final_Polars.xlsx"
    # print(f"Saving Excel: {output_path_excel}")
    # try:
    #     atd_out.write_excel(output_path_excel, worksheet="ATD_Final")
    #     print("--- \u2705 EXCEL SAVED ---")
    # except Exception as e:
    #     print(f"--- \u26a0\ufe0f EXCEL SAVE ERROR: {e} ---")

    # --- Save Parquet and CSV (parallel write) ---
    # --- Save Parquet and CSV (Sequential write) ---
    output_path_parquet = folder_paths["resources"] / "ATD_Final.parquet"
    output_path_csv     = folder_paths["resources"] / "ATD_Final.csv"

    try:
        atd_out.write_parquet(output_path_parquet, compression='zstd')
        print(f"--- ✅ PARQUET SAVED: {output_path_parquet} ---")
    except Exception as e:
        print(f"--- ❌ PARQUET SAVE ERROR: {e} ---")

    try:
        atd_out.write_csv(output_path_csv)
        print(f"--- ✅ CSV SAVED: {output_path_csv} ---")
    except Exception as e:
        print(f"--- ❌ CSV SAVE ERROR: {e} ---")

    # Show summary
    print(f"Final result: {len(atd_out)} rows")
    print("First 5 rows:")
    print(atd_out.head())

except FileNotFoundError as e:
    print(f"\n--- \u274c FILE NOT FOUND ---")
    print(f"Error: {e}")
    print("Please verify `folder_paths` settings.")
except Exception as e:
    print(f"\n--- \u274c EXECUTION ERROR ---")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

--- STARTING AUTOMATED ETL PIPELINE ---

--- STEP 1: LOAD RAW DATA ---
Loading RTA Reports from: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\STORAGE_OUTPUT_RTA
Loading HC Extend from: C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Headcount\HC Extend by Month
      hc_raw (raw): 275360 rows
      rta_raw (raw): 140079 rows
      roster_raw (from Cell 4): 13468 rows
      leave_raw (from Cell 3): 5529 rows

--- STEP 2: NORMALIZE KEY DTYPES (LAZY PARALLEL) ---
      Key dtypes normalized.

--- STEPS 3-6: RUN ETL PIPELINE ---
      Processing Roster...
      Processing HC_Extend...
      Processing Leave...

--- STEP 4: ACTIVITY PROCESSING (DAX + RENAME) ---
      Processing RTA (DAX simulation)...
            GroupBy (Date, Email Id) created...
            All DAX Measures computed.
      Activity columns renamed.

--- STEP 5: ATD_FINAL COMPUTATION ---
      Starting ATD computation (M code 'ATD_Final' simulat

In [25]:
# ==================================================================
# ===== CELL 7: PIVOT SUMMARY REPORT =====
# ==================================================================
try:
    if 'atd_out' in locals():
        print("\n--- PIVOT SUMMARY REPORT ---")

        # Ensure 'Date' is pl.Date type
        atd_out_pivot = atd_out.with_columns(
            pl.col("Date").cast(pl.Date)
        )

        # Get the most recent month
        latest_month = atd_out_pivot.select(pl.col("Month").drop_nulls().last()).item()

        if latest_month:
            print(f"Aggregating data for month: {latest_month}...\n")
            atd_month = atd_out_pivot.filter(pl.col("Month") == latest_month)

            # Group by Date and sum key metrics
            cols_to_sum = ['HC Schedule', 'Present', 'Planned', 'Unplanned']
            summary_month = atd_month.group_by("Date").agg(
                [pl.col(c).sum() for c in cols_to_sum]
            ).sort("Date")

            print(f"=== MONTHLY REPORT: {latest_month} ===")
            print(summary_month)

            print("\n=== GRAND TOTAL ===")
            print(summary_month.select(cols_to_sum).sum())

        else:
            print("No monthly data found for aggregation.")

    else:
        print("ERROR: DataFrame 'atd_out' not found.")

except Exception as e:
    print(f"\n--- \u274c AGGREGATION ERROR ---")
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()


--- PIVOT SUMMARY REPORT ---
Aggregating data for month: Feb-26...

=== MONTHLY REPORT: Feb-26 ===
shape: (28, 5)
┌────────────┬─────────────┬─────────┬─────────┬───────────┐
│ Date       ┆ HC Schedule ┆ Present ┆ Planned ┆ Unplanned │
│ ---        ┆ ---         ┆ ---     ┆ ---     ┆ ---       │
│ date       ┆ f64         ┆ f64     ┆ f64     ┆ f64       │
╞════════════╪═════════════╪═════════╪═════════╪═══════════╡
│ 2026-02-01 ┆ 58.0        ┆ 50.5    ┆ 3.0     ┆ 4.5       │
│ 2026-02-02 ┆ 77.0        ┆ 71.5    ┆ 2.0     ┆ 3.5       │
│ 2026-02-03 ┆ 75.0        ┆ 68.5    ┆ 4.0     ┆ 2.5       │
│ 2026-02-04 ┆ 69.0        ┆ 61.0    ┆ 3.0     ┆ 5.0       │
│ 2026-02-05 ┆ 65.0        ┆ 58.5    ┆ 2.0     ┆ 4.5       │
│ …          ┆ …           ┆ …       ┆ …       ┆ …         │
│ 2026-02-24 ┆ 73.0        ┆ 58.0    ┆ 6.0     ┆ 9.0       │
│ 2026-02-25 ┆ 73.0        ┆ 58.5    ┆ 5.0     ┆ 9.5       │
│ 2026-02-26 ┆ 69.0        ┆ 57.5    ┆ 5.0     ┆ 6.5       │
│ 2026-02-27 ┆ 72.0        ┆ 62

In [26]:
# ==================================================================
# ===== DEBUG CELL: FILTER UNPLANNED > 0 ON 26/11/2025 =====
# ==================================================================

target_date = pl.lit("2025-11-26").str.to_date(strict=False)

unplanned_cases = atd_out.filter(
    (pl.col("Date") == target_date) &
    (pl.col("Unplanned") > 0)
)

print(f"--- UNPLANNED > 0 ON 26/11/2025 ({unplanned_cases.height} rows) ---")

cols_to_view = [
    "OracleID", "Employee Name", "LOB", "Supervisor Name",
    "Shift Tracking", "First Shift", "Original.Shift", "Attendance",
    "HC Schedule", "Present", "Planned", "Unplanned"
]

available_cols = [c for c in cols_to_view if c in unplanned_cases.columns]

if unplanned_cases.height > 0:
    print(unplanned_cases.select(available_cols))
else:
    print("No Unplanned > 0 cases found on this date.")

# unplanned_cases.write_excel("Check_Unplanned_26Nov.xlsx")

--- UNPLANNED > 0 ON 26/11/2025 (0 rows) ---
No Unplanned > 0 cases found on this date.


In [27]:
# ==================================================================
# ===== DEBUG CELL: FIND FULL SOURCE FILE NAME =====
# ==================================================================
import polars as pl

# Filter for the problematic record
target_name = "DOAN AN NY"
target_date = pl.lit("2026-05-01").str.to_date(strict=False)

# Look up Email from HC (to cross-reference in Roster)
if 'hc_raw' in locals():
    emp_info = hc_raw.filter(pl.col("Employee Name") == target_name).select("Email Id").head(1)
    if emp_info.height > 0:
        target_email = emp_info.item(0, "Email Id")

        # Filter in Roster Raw
        wrong_data = roster_raw.filter(
            (pl.col("Scheduled_Date") == target_date) &
            (pl.col("Email Id") == target_email)
        ).select(["Source_File", "Scheduled_Date", "Shift"])

        # Print full file name (no truncation)
        print(f"--- LOOKUP RESULT FOR: {target_name} ---")

        if wrong_data.height > 0:
            for row in wrong_data.iter_rows(named=True):
                print("\n[RECORD FOUND]")
                print(f"\U0001f4c2 SOURCE FILE:  {row['Source_File']}")
                print(f"\U0001f4c5 DATE:         {row['Scheduled_Date']}")
                print(f"\u23f0 SHIFT:        {row['Shift']}")
                print("-" * 50)
        else:
            print("No matching record found in roster_raw for this date and employee.")

    else:
        print(f"Employee {target_name} not found in HC table.")
else:
    print("Variable hc_raw not available.")

--- LOOKUP RESULT FOR: DOAN AN NY ---

[RECORD FOUND]
📂 SOURCE FILE:  Master_Schedule_Merged.xlsx
📅 DATE:         2026-05-01
⏰ SHIFT:        HO
--------------------------------------------------
